I am trying to extract all errors LLM generated, initially i was excluding specific error types such as MOJOException etc.

In [ ]:
import json
import csv
import re
from pathlib import Path
from typing import Dict, List


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# CSV file with BUMP ground-truth breaking change metadata (shared across all variants)
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Output CSV — always created as a NEW file with all 9 variants
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/detected_bc_errortype_coverage.csv"

# All 9 (model, context_variant) configurations
VARIANTS = [
    # GPT-4o
    {
        'model': 'GPT-4o',
        'context_variant': 'Minimal',
        'llm_results_json': '/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs',
    },
    {
        'model': 'GPT-4o',
        'context_variant': 'Method',
        'llm_results_json': '/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/logs',
    },
    {
        'model': 'GPT-4o',
        'context_variant': 'Class',
        'llm_results_json': '/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/logs',
    },
    # Qwen-480B
    {
        'model': 'Qwen-480B',
        'context_variant': 'Minimal',
        'llm_results_json': '/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/logs',
    },
    {
        'model': 'Qwen-480B',
        'context_variant': 'Method',
        'llm_results_json': '/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/logs',
    },
    {
        'model': 'Qwen-480B',
        'context_variant': 'Class',
        'llm_results_json': '/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/logs',
    },
    # GPT-OSS-120b
    {
        'model': 'GPT-OSS-120b',
        'context_variant': 'Minimal',
        'llm_results_json': '/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/logs',
    },
    {
        'model': 'GPT-OSS-120b',
        'context_variant': 'Method',
        'llm_results_json': '/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/logs',
    },
    {
        'model': 'GPT-OSS-120b',
        'context_variant': 'Class',
        'llm_results_json': '/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/logs',
    },
]

# ──────────────────────────────────────────────────────────────────────────────


class LogParser:
    """
    Parse Maven/Java test logs and extract all error information.
    Same logic as Script 2 (bump_executor.py) LogParser.
    """

    def __init__(self, log_text: str):
        self.log = log_text

    def parse(self) -> Dict:
        """Parse log and extract all error information."""
        return {
            'all_exceptions': self.extract_exceptions(),
            'all_errors': self.extract_errors(),
            'failed_tests': self.extract_failed_tests(),
            'compilation_failures': self.extract_compilation_failures(),
            'error_messages': self.extract_error_messages(),
            'stack_traces': self.extract_stack_traces(),
            'maven_errors': self.extract_maven_errors(),
            'test_summary': self.extract_test_summary(),
        }

    def extract_exceptions(self) -> List[Dict]:
        """Extract all Java exceptions from the log."""
        exceptions = []

        # Pattern for Java exceptions
        pattern = r'([\w.]+(?:Exception|Error))(?::\s*(.+?))?(?=\n|\r|$)'

        for match in re.finditer(pattern, self.log):
            exception_type = match.group(1)
            exception_message = match.group(2).strip() if match.group(2) else ""

            # Get line number
            line_num = self.log[:match.start()].count('\n') + 1

            # Get context (50 chars before and after)
            start = max(0, match.start() - 50)
            end = min(len(self.log), match.end() + 100)
            context = self.log[start:end].replace('\n', ' ').strip()

            exceptions.append({
                'type': exception_type,
                'message': exception_message[:200],
                'line': line_num,
                'context': context[:200]
            })

        return exceptions

    def extract_errors(self) -> List[str]:
        """Extract all [ERROR] lines from Maven output."""
        errors = []
        pattern = r'\[ERROR\]\s*(.+?)(?=\n|$)'

        for match in re.finditer(pattern, self.log, re.MULTILINE):
            error_text = match.group(1).strip()
            if error_text and len(error_text) > 5:
                errors.append(error_text[:300])

        return list(dict.fromkeys(errors))

    def extract_failed_tests(self) -> List[Dict]:
        """Extract information about failed tests."""
        failed_tests = []

        # Pattern 1: JUnit format - TestName(ClassName)
        pattern1 = r'(\w+)\(([\w.]+)\)\s+Time elapsed:.*?<<<\s*(FAILURE|ERROR)!'
        for match in re.finditer(pattern1, self.log):
            failed_tests.append({
                'test_method': match.group(1),
                'test_class': match.group(2),
                'failure_type': match.group(3),
                'format': 'junit'
            })

        # Pattern 2: Maven format - package.Class.method
        pattern2 = r'Failed tests?:\s+([\w.]+\.[\w.]+)'
        for match in re.finditer(pattern2, self.log):
            full_name = match.group(1)
            parts = full_name.rsplit('.', 1)
            failed_tests.append({
                'test_method': parts[1] if len(parts) > 1 else full_name,
                'test_class': parts[0] if len(parts) > 1 else 'Unknown',
                'failure_type': 'FAILURE',
                'format': 'maven'
            })

        # Pattern 3: Errors in tests
        pattern3 = r'Errors?:\s+([\w.]+\.[\w.]+)'
        for match in re.finditer(pattern3, self.log):
            full_name = match.group(1)
            parts = full_name.rsplit('.', 1)
            failed_tests.append({
                'test_method': parts[1] if len(parts) > 1 else full_name,
                'test_class': parts[0] if len(parts) > 1 else 'Unknown',
                'failure_type': 'ERROR',
                'format': 'maven'
            })

        return failed_tests

    def extract_compilation_failures(self) -> List[Dict]:
        """Extract compilation errors."""
        compilation_errors = []

        if re.search(r'COMPILATION ERROR', self.log, re.IGNORECASE):
            pattern = r'\[ERROR\]\s*([\w/\\.]+\.java):\[(\d+),(\d+)\]\s*(.+?)(?=\n|$)'
            for match in re.finditer(pattern, self.log):
                compilation_errors.append({
                    'file': match.group(1),
                    'line': match.group(2),
                    'column': match.group(3),
                    'error': match.group(4).strip()[:200]
                })

        return compilation_errors

    def extract_error_messages(self) -> List[str]:
        """Extract readable error messages."""
        messages = []

        pattern1 = r'expected:\s*<?(.+?)>?\s*but was:\s*<?(.+?)>?'
        for match in re.finditer(pattern1, self.log, re.IGNORECASE):
            messages.append(f"Expected: {match.group(1)}, but was: {match.group(2)}")

        pattern2 = r'(?:Exception|Error):\s*([^\n]{20,200})'
        for match in re.finditer(pattern2, self.log):
            msg = match.group(1).strip()
            if msg not in str(messages):
                messages.append(msg)

        return messages[:10]

    def extract_stack_traces(self) -> List[str]:
        """Extract stack traces."""
        stack_traces = []

        pattern = r'((?:[\w.]+(?:Exception|Error)[^\n]*(?:\n\s+at\s+[\w.$<>]+\([^\)]+\))+))'

        matches = re.finditer(pattern, self.log, re.MULTILINE)
        for match in matches:
            trace = match.group(1)
            lines = trace.split('\n')[:6]
            stack_traces.append('\n'.join(lines))

        return stack_traces[:3]

    def extract_maven_errors(self) -> Dict:
        """Extract Maven-specific error information."""
        maven_info = {
            'build_failure': bool(re.search(r'BUILD FAILURE', self.log)),
            'build_success': bool(re.search(r'BUILD SUCCESS', self.log)),
            'reactor_summary': None,
            'failure_message': None
        }

        failure_pattern = r'(?:Failure|Error)\s*message:\s*(.+?)(?=\n|$)'
        match = re.search(failure_pattern, self.log, re.IGNORECASE)
        if match:
            maven_info['failure_message'] = match.group(1).strip()

        return maven_info

    def extract_test_summary(self) -> Dict:
        """Extract test execution summary."""
        summary = {
            'tests_run': 0,
            'failures': 0,
            'errors': 0,
            'skipped': 0
        }

        pattern = r'Tests run:\s*(\d+).*?Failures:\s*(\d+).*?Errors:\s*(\d+).*?Skipped:\s*(\d+)'
        match = re.search(pattern, self.log)

        if match:
            summary['tests_run'] = int(match.group(1))
            summary['failures'] = int(match.group(2))
            summary['errors'] = int(match.group(3))
            summary['skipped'] = int(match.group(4))

        return summary


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split.
    NO exclusions — all exception/error types are kept.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e:
            continue
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Run the full LogParser (same as Script 2) on a test execution log file
    and return all unique short exception/error class names found.
    Uses extract_exceptions() which captures every Exception/Error with
    no exclusions.
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()

    parser = LogParser(content)
    parsed = parser.parse()

    found = set()
    for exc in parsed['all_exceptions']:
        full_name = exc['type']
        short_name = full_name.split('.')[-1]
        if short_name:
            found.add(short_name)

    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def process_variant(variant: dict, bump: dict) -> list:
    """
    Process a single (model, context_variant) configuration.
    Returns a list of row dicts for the output CSV.
    """
    model_name = variant['model']
    context_variant = variant['context_variant']
    llm_results_json = variant['llm_results_json']
    llm_logs_dir = variant['llm_logs_dir']

    print(f"\n{'='*70}")
    print(f"Processing: {model_name} | {context_variant}")
    print(f"  JSON: {llm_results_json}")
    print(f"  Logs: {llm_logs_dir}")
    print(f"{'='*70}")

    llm = load_llm(llm_results_json)
    logs = Path(llm_logs_dir)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"  Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        # Skip transplant_issue tests, process all remaining test log files
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               model_name,
            'context_variant':     context_variant,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

    print(f"  Done. {len(rows)} detected instances for {model_name} | {context_variant}")
    return rows


def main():
    # Load BUMP ground truth once (shared across all variants)
    bump = load_bump(BUMP_CSV)
    print(f"Loaded BUMP CSV: {len(bump)} instances")

    all_rows = []

    for variant in VARIANTS:
        rows = process_variant(variant, bump)
        all_rows.extend(rows)

    # Write CSV — always create a NEW file with all variants
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n{'='*70}")
    print(f"ALL DONE. {len(all_rows)} total rows written to {OUTPUT_CSV}")
    print(f"{'='*70}")
    for variant in VARIANTS:
        count = sum(1 for r in all_rows if r['model'] == variant['model'] and r['context_variant'] == variant['context_variant'])
        print(f"  {variant['model']} | {variant['context_variant']}: {count} instances")


if __name__ == "__main__":
    main()

Loaded BUMP CSV: 169 instances

Processing: GPT-4o | Minimal
  JSON: /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json
  Logs: /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs
  Done. 17 detected instances for GPT-4o | Minimal

Processing: GPT-4o | Method
  JSON: /Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json
  Logs: /Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/logs
  Done. 13 detected instances for GPT-4o | Method

Processing: GPT-4o | Class
  JSON: /Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json
  Logs: /Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/logs
  Done. 27 detected instances for GPT-4o | Class

Processing: Qwen-480B | Minimal
  JSON: /Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json
  Logs: /Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/logs
  Don

In [9]:
"""
add_error_category_to_bump_v2.py
=================================
Adds normalized_error_category column to BUMP CSV.
Reads exception_types, extracts short class names, keeps ALL error types
with NO exclusions and NO hardcoded mapping.
Each unique exception/error class name becomes its own category.
Writes to a new v2 file to avoid corrupting the original.
"""

import csv
import pandas as pd
from collections import Counter


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV   = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"
OUTPUT_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories_v2.csv"
# ──────────────────────────────────────────────────────────────────────────────


def categorize_exceptions(raw: str) -> str:
    """
    Extract all exception/error short class names from the pipe-separated
    exception_types column. No exclusions, no hardcoded mapping.
    Each unique short class name is kept as-is.
    Returns pipe-separated string of unique short class names, or empty string
    if none found.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return ''

    categories = []
    for e in str(raw).split('|'):
        # Strip package prefix to get short class name
        short = e.strip().split('.')[-1]
        if not short:
            continue
        # Keep every unique short class name — no exclusions
        if short not in categories:
            categories.append(short)

    return '|'.join(categories)


def detect_sep(path: str) -> str:
    with open(path, encoding='utf-8') as f:
        header = f.readline()
    tabs   = header.count('\t')
    commas = header.count(',')
    sep = '\t' if tabs > commas else ','
    print(f"  Separator: {'TAB' if sep == chr(9) else 'COMMA'} (tabs={tabs}, commas={commas})")
    return sep


def main():
    sep = detect_sep(BUMP_CSV)
    df  = pd.read_csv(BUMP_CSV, sep=sep, dtype=str, keep_default_na=False,
                      quoting=csv.QUOTE_MINIMAL)
    print(f"  Loaded {len(df)} rows, {len(df.columns)} columns")
    print(f"  Columns: {list(df.columns)}")

    # Debug: show raw exception_types for first 5 rows
    print(f"\n  Sample exception_types (first 5 rows):")
    for _, row in df.head(5).iterrows():
        raw = row.get('exception_types', '')
        cat = categorize_exceptions(raw)
        print(f"    {row['custom_id']:8s}  raw: {raw[:80]}")
        print(f"    {'':8s}  cat: {cat}")

    # Add normalized column
    df['normalized_error_category'] = df['exception_types'].apply(categorize_exceptions)

    # Summary: count how many instances each unique error type appears in
    cat_counts: Counter = Counter()
    empty_count = 0
    for val in df['normalized_error_category']:
        if not val:
            empty_count += 1
            continue
        for c in str(val).split('|'):
            c = c.strip()
            if c:
                cat_counts[c] += 1

    print(f"\n  All unique error types ({len(cat_counts)} types):")
    for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
        print(f"    {cat:50s}  {cnt:3d} instances")
    if empty_count:
        print(f"    {'(no exception_types)':50s}  {empty_count:3d} instances")

    # Write to new file, same separator
    df.to_csv(OUTPUT_CSV, sep=sep, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"\n  Saved → {OUTPUT_CSV}  ({len(df.columns)} columns, {len(df)} rows)")
    print(f"  Original {BUMP_CSV} is untouched.")


if __name__ == "__main__":
    main()

  Separator: COMMA (tabs=0, commas=30)
  Loaded 89 rows, 31 columns
  Columns: ['custom_id', 'clientGithubURL', 'clientProject', 'clientProjectOrganisation', 'breakingCommit', 'dependencyGroupID', 'dependencyArtifactID', 'previousVersion', 'newVersion', 'failureCategory', 'docker_image_breaking', 'execution_timestamp', 'execution_success', 'return_code', 'execution_time_seconds', 'tests_run', 'test_failures', 'test_errors', 'test_skipped', 'num_exceptions', 'num_failed_tests', 'num_compilation_errors', 'num_error_messages', 'exception_types', 'first_error_message', 'all_maven_errors', 'build_status', 'error_category', 'notes', 'log_file', 'parsed_errors_file']

  Sample exception_types (first 5 rows):
    BBC01     raw: java.lang.NoClassDefFoundError|StopException|java.lang.ClassNotFoundException|Mo
              cat: NoClassDefFoundError|StopException|ClassNotFoundException|MojoFailureException
    BBC02     raw: java.lang.NoClassDefFoundError|java.lang.RuntimeException|java.lang.Clas

Important data analysis investigation for RQ2 successful cases...

Table VI

In [9]:
"""
analysis1_error_type_coverage_v7.py
====================================
Pipeline:
  1. BUMP CSV      → reads normalized_error_category column
  2. Detected CSV  → per (model, context_variant, instance)
  3. For each error type x instance x model x variant, classify:

Classification of each (instance, error_type, model, variant) combination:

  detected    : BUMP ground truth for THIS instance has this error type
                AND LLM triggered this error type for this instance.
                → True positive: LLM correctly detected the breaking change.

  missed      : BUMP ground truth for THIS instance has this error type
                AND LLM did NOT trigger it.
                → False negative: LLM missed the actual breaking change error.

  mismatched  : BUMP ground truth for THIS instance does NOT have this error type,
                BUT LLM triggered it, AND this error type EXISTS in BUMP for
                OTHER instances.
                → LLM triggered a real Java exception but for the wrong instance.

  novel       : LLM triggered this error type AND it does NOT exist in BUMP
                for ANY instance at all.
                → LLM generated a completely new error type not seen in BUMP.
                  Example: test method names like test_handleRequestWithIOException
                  that LLM generates as test names but are not Java exceptions.

Note:
  - detected + missed covers all cases where BUMP has this error type for this instance
  - mismatched + novel covers all cases where LLM triggered something not in BUMP
    ground truth for this instance
  - novel specifically means the error type is absent from ALL of BUMP
"""

import sys
import pandas as pd
from collections import defaultdict
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV           = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories_v2.csv"
DETECTED_CSV       = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage_v2.csv"
OUTPUT_DETAIL_CSV  = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_detail_v7.csv"
OUTPUT_SUMMARY_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_summary_v7.csv"
OUTPUT_TXT         = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_report_v7.txt"

MODEL_ORDER   = ['GPT-4o', 'Qwen-480B', 'GPT-OSS-120b']
VARIANT_ORDER = ['Minimal', 'Method', 'Class']

RARE_THRESHOLD = 2

pd.set_option('display.max_rows',     500)
pd.set_option('display.max_columns',  50)
pd.set_option('display.width',        200)
pd.set_option('display.max_colwidth', 100)
# ──────────────────────────────────────────────────────────────────────────────


# ─── Tee: write to both console and file ─────────────────────────────────────

class Tee:
    """Writes to both console and a file simultaneously."""
    def __init__(self, filepath):
        self.console = sys.stdout
        self.file    = open(filepath, 'w', encoding='utf-8')

    def write(self, msg):
        self.console.write(msg)
        self.file.write(msg)

    def flush(self):
        self.console.flush()
        self.file.flush()

    def close(self):
        self.file.close()


# ─── Helpers ─────────────────────────────────────────────────────────────────

def parse_pipe(val) -> set:
    """Pipe-separated string → set of short class names."""
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {e.strip().split('.')[-1] for e in str(val).split('|') if e.strip()}


def dedup_ids(series) -> str:
    """Deduplicate pipe-separated IDs across multiple rows."""
    all_ids = set()
    for val in series:
        if val and str(val).strip() not in ('', 'nan'):
            for v in str(val).split('|'):
                v = v.strip()
                if v:
                    all_ids.add(v)
    return '|'.join(sorted(all_ids))


def print_section(title, description, df_section, row_printer):
    print("\n" + "=" * 80)
    print(f"{title} ({len(df_section)} error types)")
    print(f"  {description}")
    print("=" * 80)
    if len(df_section) == 0:
        print("  (none)")
        return
    for _, row in df_section.iterrows():
        row_printer(row)


# ─── STEP 1: Load BUMP CSV ────────────────────────────────────────────────────

def load_bump(bump_csv: str, detected_ids: set):
    """
    Returns:
      bump_cats_per_instance : dict[custom_id → set of error types]
      bump_error_types_global: set of ALL error types anywhere in BUMP
      common_cats            : error types with bump_total >= RARE_THRESHOLD
      rare_cats              : error types with bump_total < RARE_THRESHOLD
      bump_total_by_cat      : dict[error_type → count]
    """
    df = pd.read_csv(bump_csv, sep=None, engine='python',
                     dtype=str, keep_default_na=False)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    bump_cats_per_instance: dict = {}
    all_counts: dict = defaultdict(int)

    for _, row in df.iterrows():
        cid  = row['custom_id']
        cats = parse_pipe(row.get('normalized_error_category', ''))
        for c in cats:
            all_counts[c] += 1
        if cid in detected_ids:
            bump_cats_per_instance[cid] = cats

    # All error types that appear anywhere in BUMP
    bump_error_types_global = set(all_counts.keys())

    all_cats_sorted = sorted(all_counts.keys(), key=lambda c: -all_counts[c])
    common_cats = [c for c in all_cats_sorted if all_counts[c] >= RARE_THRESHOLD]
    rare_cats   = [c for c in all_cats_sorted if all_counts[c] < RARE_THRESHOLD]

    print(f"  BUMP CSV: {len(df)} instances")
    print(f"  Total unique error types in BUMP: {len(bump_error_types_global)}")
    print(f"  Common error types (n>={RARE_THRESHOLD}): {len(common_cats)}")
    for c in common_cats:
        print(f"    {c:50s}  n={all_counts[c]}")
    print(f"  Rare error types (n<{RARE_THRESHOLD}): {len(rare_cats)}")
    for c in rare_cats:
        print(f"    {c:50s}  n={all_counts[c]}")

    return (bump_cats_per_instance, bump_error_types_global,
            common_cats, rare_cats, dict(all_counts))


# ─── STEP 2: Load detected CSV ────────────────────────────────────────────────

def load_detected(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine='python',
                     dtype=str, keep_default_na=False)
    df['custom_id']  = df['custom_id'].astype(str).str.strip()
    df['_bump_cats'] = df['bump_bc_errors'].apply(parse_pipe)
    df['_llm_cats']  = df['llm_detected_errors'].apply(parse_pipe)
    print(f"  Detected CSV: {len(df)} rows | "
          f"{df['custom_id'].nunique()} unique instances | "
          f"{df['model'].nunique()} models | "
          f"{df['context_variant'].nunique()} variants")
    return df


# ─── STEP 3: Find novel error types ──────────────────────────────────────────

def find_novel_types(df: pd.DataFrame,
                     bump_error_types_global: set) -> set:
    """
    Find all error types LLM generated that do not exist anywhere in BUMP.
    These are truly novel — not just mismatched instances.
    """
    llm_all_types = set()
    for val in df['_llm_cats']:
        llm_all_types |= val

    novel = llm_all_types - bump_error_types_global
    print(f"\n  Novel error types (in LLM but not in BUMP at all): {len(novel)}")
    for t in sorted(novel):
        # Count how many times it appears
        count = sum(1 for cats in df['_llm_cats'] if t in cats)
        print(f"    {t:50s}  triggered in {count} rows")
    return novel


# ─── STEP 4: Build detail table ───────────────────────────────────────────────

def build_detail_table(
    bump_total_by_cat: dict,
    bump_cats_per_instance: dict,
    bump_error_types_global: set,
    novel_types: set,
    all_cats: list,
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    One row per (model, context_variant, bump_error_type, custom_id)
    where something is happening (in_bump=1 OR llm_triggered=1).

    Also includes novel error types that LLM generated but don't exist in BUMP.
    For novel types: bump_total=0, in_bump_instance=0, in_bump_global=0.
    """
    # All categories to consider = BUMP cats + novel LLM cats
    all_cats_extended = all_cats + sorted(novel_types)

    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) &
                     (df['context_variant'] == variant)]

            for _, row in sub.iterrows():
                cid               = row['custom_id']
                bump_errs_instance = bump_cats_per_instance.get(cid, set())
                llm_errs          = row['_llm_cats']

                # All error types relevant to this row
                all_relevant = bump_errs_instance | llm_errs

                for cat in all_relevant:
                    in_bump_instance = int(cat in bump_errs_instance)
                    llm_triggered    = int(cat in llm_errs)
                    in_bump_global   = int(cat in bump_error_types_global)

                    # Classify — exactly one of these will be 1
                    detected   = int(in_bump_instance == 1 and llm_triggered == 1)
                    missed     = int(in_bump_instance == 1 and llm_triggered == 0)
                    mismatched = int(in_bump_instance == 0 and llm_triggered == 1
                                     and in_bump_global == 1)
                    novel      = int(in_bump_instance == 0 and llm_triggered == 1
                                     and in_bump_global == 0)

                    records.append({
                        'model':             model,
                        'context_variant':   variant,
                        'bump_error_type':   cat,
                        'custom_id':         cid,
                        'bump_total':        bump_total_by_cat.get(cat, 0),
                        'in_bump_instance':  in_bump_instance,
                        'in_bump_global':    in_bump_global,
                        'llm_triggered':     llm_triggered,
                        'detected':          detected,
                        'missed':            missed,
                        'mismatched':        mismatched,
                        'novel':             novel,
                    })

    df_out = pd.DataFrame(records)

    # Sanity check
    check = df_out[['detected', 'missed', 'mismatched', 'novel']].sum(axis=1)
    assert (check == 1).all(), \
        f"ERROR: {(check != 1).sum()} rows have incorrect classification!"
    print("  Sanity check passed: every row has exactly one classification.")

    return df_out


# ─── STEP 5: Build summary from detail table ──────────────────────────────────

def build_summary(
    detail: pd.DataFrame,
    all_cats: list,
    novel_types: set,
    bump_total_by_cat: dict,
) -> pd.DataFrame:
    """
    Aggregates detail table per (bump_error_type, model, context_variant).
    Includes novel error types as separate rows with bump_total=0.
    """
    all_cats_extended = all_cats + sorted(novel_types)
    records = []

    for cat in all_cats_extended:
        bump_total = bump_total_by_cat.get(cat, 0)
        cat_sub    = detail[detail['bump_error_type'] == cat]

        # Unique instances detected by ANY model-variant
        detected_any_ids = set(
            cat_sub[cat_sub['detected'] == 1]['custom_id'].unique()
        )

        for model in MODEL_ORDER:
            for variant in VARIANT_ORDER:
                mv_sub = cat_sub[
                    (cat_sub['model'] == model) &
                    (cat_sub['context_variant'] == variant)
                ]

                detected   = int(mv_sub['detected'].sum())
                missed     = int(mv_sub['missed'].sum())
                mismatched = int(mv_sub['mismatched'].sum())
                novel      = int(mv_sub['novel'].sum())
                rate       = round(detected / bump_total * 100, 1) \
                             if bump_total > 0 else 0.0

                detected_ids   = '|'.join(sorted(
                    mv_sub[mv_sub['detected'] == 1]['custom_id'].tolist()))
                mismatched_ids = '|'.join(sorted(
                    mv_sub[mv_sub['mismatched'] == 1]['custom_id'].tolist()))
                novel_ids      = '|'.join(sorted(
                    mv_sub[mv_sub['novel'] == 1]['custom_id'].tolist()))

                records.append({
                    'bump_error_type':   cat,
                    'bump_total':        bump_total,
                    'in_bump_global':    int(cat not in novel_types),
                    'model':             model,
                    'context_variant':   variant,
                    'detected':          detected,
                    'missed':            missed,
                    'mismatched':        mismatched,
                    'novel':             novel,
                    'detection_rate_%':  rate,
                    'detected_ids':      detected_ids,
                    'mismatched_ids':    mismatched_ids,
                    'novel_ids':         novel_ids,
                    'detected_by_any':   len(detected_any_ids),
                    'detected_any_ids':  '|'.join(sorted(detected_any_ids)),
                })

    return pd.DataFrame(records)


# ─── STEP 6: Summarise and report ────────────────────────────────────────────

def summarise_and_report(summary: pd.DataFrame, novel_types: set):
    """
    Reads summary DataFrame and produces a structured report.
    """
    num_cols = ['bump_total', 'detected', 'missed', 'mismatched',
                'novel', 'detection_rate_%', 'detected_by_any',
                'in_bump_global']
    for c in num_cols:
        summary[c] = pd.to_numeric(summary[c], errors='coerce').fillna(0).astype(int)

    # Overall per error type
    overall = (
        summary.groupby('bump_error_type')
        .agg(
            bump_total       = ('bump_total',      'first'),
            in_bump_global   = ('in_bump_global',  'first'),
            detected_by_any  = ('detected_by_any', 'first'),
            detected_any_ids = ('detected_any_ids','first'),
            total_mismatched = ('mismatched',      'sum'),
            total_novel      = ('novel',           'sum'),
            total_missed     = ('missed',          'sum'),
        )
        .reset_index()
        .sort_values('bump_total', ascending=False)
    )

    # Best detection per model
    for model in MODEL_ORDER:
        best = (
            summary[summary['model'] == model]
            .groupby('bump_error_type')['detected']
            .max()
            .reset_index()
            .rename(columns={'detected': f'{model}_best'})
        )
        overall = overall.merge(best, on='bump_error_type', how='left')

    # Best detection rate
    best_rate = (
        summary.groupby('bump_error_type')['detection_rate_%']
        .max()
        .reset_index()
        .rename(columns={'detection_rate_%': 'best_rate_%'})
    )
    overall = overall.merge(best_rate, on='bump_error_type', how='left')

    # Deduplicated mismatched IDs
    mismatch_ids = (
        summary[summary['mismatched'] > 0]
        .groupby('bump_error_type')['mismatched_ids']
        .apply(dedup_ids)
        .reset_index()
        .rename(columns={'mismatched_ids': 'mismatched_unique_ids'})
    )
    overall = overall.merge(mismatch_ids, on='bump_error_type', how='left')

    # Deduplicated novel IDs
    novel_ids_agg = (
        summary[summary['novel'] > 0]
        .groupby('bump_error_type')['novel_ids']
        .apply(dedup_ids)
        .reset_index()
        .rename(columns={'novel_ids': 'novel_unique_ids'})
    )
    overall = overall.merge(novel_ids_agg, on='bump_error_type', how='left')

    # Split into groups
    detected        = overall[overall['detected_by_any'] > 0].copy()
    mismatched_only = overall[
        (overall['detected_by_any'] == 0) &
        (overall['total_mismatched'] > 0) &
        (overall['in_bump_global'] == 1)
    ].copy()
    missed_only     = overall[
        (overall['detected_by_any'] == 0) &
        (overall['total_mismatched'] == 0) &
        (overall['total_novel'] == 0) &
        (overall['total_missed'] > 0) &
        (overall['in_bump_global'] == 1)
    ].copy()
    zero            = overall[
        (overall['detected_by_any'] == 0) &
        (overall['total_mismatched'] == 0) &
        (overall['total_novel'] == 0) &
        (overall['total_missed'] == 0) &
        (overall['in_bump_global'] == 1)
    ].copy()
    novel_only      = overall[
        (overall['in_bump_global'] == 0)
    ].copy()

    # Row printers
    def print_detected(row):
        ids = sorted(v for v in str(row.get('detected_any_ids', '')).split('|') if v.strip())
        print(f"\n  {row['bump_error_type']}")
        print(f"    BUMP instances : {row['bump_total']}")
        print(f"    Detected (any) : {row['detected_by_any']} "
              f"({row['best_rate_%']}% best rate)")
        print(f"    GPT-4o best    : {row['GPT-4o_best']}")
        print(f"    Qwen best      : {row['Qwen-480B_best']}")
        print(f"    GPT-OSS best   : {row['GPT-OSS-120b_best']}")
        print(f"    Detected IDs   : {', '.join(ids) if ids else '--'}")

    def print_mismatched(row):
        ids = sorted(v for v in
                     str(row.get('mismatched_unique_ids', '')).split('|')
                     if v.strip())
        print(f"\n  {row['bump_error_type']}")
        print(f"    BUMP instances         : {row['bump_total']}")
        print(f"    Unique wrong instances : {len(ids)}")
        print(f"    Mismatched IDs         : {', '.join(ids) if ids else '--'}")

    def print_missed(row):
        print(f"\n  {row['bump_error_type']}")
        print(f"    BUMP instances : {row['bump_total']}")
        print(f"    Total missed   : {row['total_missed']} "
              f"(LLM never triggered this error type)")

    def print_zero(row):
        print(f"  {row['bump_error_type']:50s}  n={row['bump_total']}")

    def print_novel(row):
        ids = sorted(v for v in
                     str(row.get('novel_unique_ids', '')).split('|')
                     if v.strip())
        print(f"\n  {row['bump_error_type']}")
        print(f"    Total rows triggered : {row['total_novel']}")
        print(f"    Instance IDs         : {', '.join(ids) if ids else '--'}")

    # Print all sections
    print_section(
        "GROUP 1: DETECTED",
        "LLM correctly detected at least one instance of this error type.",
        detected, print_detected
    )

    print_section(
        "GROUP 2: MISMATCHED ONLY",
        "LLM triggered this error type but for different instances than BUMP.\n"
        "  These are real Java exception types that exist in BUMP, but LLM\n"
        "  triggered them for wrong instances (not the ground truth instance).",
        mismatched_only, print_mismatched
    )

    print_section(
        "GROUP 3: MISSED ONLY",
        "BUMP has these error types but LLM never triggered them in any test.",
        missed_only, print_missed
    )

    print_section(
        "GROUP 4: ZERO SIGNAL",
        "LLM had zero contact with these instances (no passing tests generated).",
        zero, print_zero
    )

    print_section(
        "GROUP 5: NOVEL (not in BUMP at all)",
        "LLM generated these error types but they do not exist anywhere in BUMP.\n"
        "  These are typically test method names or internal exceptions that\n"
        "  LLM generated as side effects, not real breaking change signals.",
        novel_only, print_novel
    )

    # Per model-variant breakdown for detected types
    print("\n" + "=" * 80)
    print("PER MODEL-VARIANT BREAKDOWN (detected error types only)")
    print("=" * 80)
    for cat in detected['bump_error_type'].tolist():
        bump_n = int(overall[overall['bump_error_type'] == cat]['bump_total'].values[0])
        print(f"\n  {cat}  (BUMP n={bump_n})")
        print(f"  {'Model':<15} {'Variant':<10} {'Detected':>10} {'Missed':>8} "
              f"{'Mismatched':>12} {'Rate%':>8}  Detected IDs")
        print(f"  {'-'*15} {'-'*10} {'-'*10} {'-'*8} {'-'*12} {'-'*8}  {'-'*40}")
        sub = summary[summary['bump_error_type'] == cat]
        for model in MODEL_ORDER:
            for variant in VARIANT_ORDER:
                row = sub[(sub['model'] == model) &
                          (sub['context_variant'] == variant)]
                if len(row) == 0:
                    continue
                row = row.iloc[0]
                det_ids = sorted(
                    v for v in str(row.get('detected_ids', '')).split('|')
                    if v.strip()
                )
                print(f"  {model:<15} {variant:<10} "
                      f"{row['detected']:>10} "
                      f"{row['missed']:>8} "
                      f"{row['mismatched']:>12} "
                      f"{row['detection_rate_%']:>8}  "
                      f"{', '.join(det_ids) if det_ids else '--'}")

    # Overall stats
    print("\n" + "=" * 80)
    print("OVERALL STATS")
    print("=" * 80)
    print(f"  Total unique error types in BUMP    : {len(overall[overall['in_bump_global']==1])}")
    print(f"  Novel error types (not in BUMP)     : {len(novel_only)}")
    print(f"  Error types with any detection      : {len(detected)}")
    print(f"  Error types with zero detection     : "
          f"{len(overall[overall['in_bump_global']==1]) - len(detected)}")
    print(f"    of which mismatched only          : {len(mismatched_only)}")
    print(f"    of which missed only              : {len(missed_only)}")
    print(f"    of which zero signal              : {len(zero)}")

    return overall


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    print("\nStep 1: Loading detected CSV...")
    df = load_detected(DETECTED_CSV)

    print("\nStep 2: Loading BUMP CSV...")
    detected_ids = set(df['custom_id'].unique())
    (bump_cats_per_instance, bump_error_types_global,
     common_cats, rare_cats, bump_total_by_cat) = load_bump(
        BUMP_CSV, detected_ids=detected_ids)

    print("\nStep 3: Finding novel error types...")
    novel_types = find_novel_types(df, bump_error_types_global)

    all_cats = common_cats + rare_cats

    print("\nStep 4: Building detail table...")
    detail = build_detail_table(
        bump_total_by_cat,
        bump_cats_per_instance,
        bump_error_types_global,
        novel_types,
        all_cats,
        df
    )
    Path(OUTPUT_DETAIL_CSV).parent.mkdir(parents=True, exist_ok=True)
    detail.to_csv(OUTPUT_DETAIL_CSV, index=False)
    print(f"  Detail table saved → {OUTPUT_DETAIL_CSV}")
    print(f"  Detail table shape: {detail.shape}")

    print("\nStep 5: Building summary table...")
    summary = build_summary(detail, all_cats, novel_types, bump_total_by_cat)
    summary.to_csv(OUTPUT_SUMMARY_CSV, index=False)
    print(f"  Summary table saved → {OUTPUT_SUMMARY_CSV}")

    print("\nStep 6: Generating report...")
    overall = summarise_and_report(summary, novel_types)

    return overall


if __name__ == "__main__":
    Path(OUTPUT_TXT).parent.mkdir(parents=True, exist_ok=True)
    tee = Tee(OUTPUT_TXT)
    sys.stdout = tee
    try:
        overall = main()
    finally:
        sys.stdout = tee.console
        tee.close()
        print(f"\nReport written to: {OUTPUT_TXT}")


Step 1: Loading detected CSV...
  Detected CSV: 137 rows | 32 unique instances | 3 models | 3 variants

Step 2: Loading BUMP CSV...
  BUMP CSV: 89 instances
  Total unique error types in BUMP: 56
  Common error types (n>=2): 24
    MojoFailureException                                n=88
    NoClassDefFoundError                                n=35
    ClassNotFoundException                              n=30
    SocketTimeoutException                              n=15
    ClassCastException                                  n=10
    AssertionError                                      n=10
    RuntimeException                                    n=8
    ExceptionInInitializerError                         n=8
    UnsupportedClassVersionError                        n=8
    BeanInstantiationException                          n=8
    NoSuchMethodError                                   n=8
    AssertionFailedError                                n=7
    UnsupportedOperationException            